# Thêm Thư Viện

In [2]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [3]:
conn_libol = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;'  # Địa chỉ IP của SQL Server
    'DATABASE=libol;'         # Tên cơ sở dữ liệu
    'UID=itc;'                # Tên đăng nhập
    'PWD=spkt@2025;'
)
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ SQL Server

In [4]:
query_trinhdo = "SELECT trinh_do_id, dbo.DecodeUTF8String(loai_trinh_do) AS Trinh_do FROM Trinh_do "
df_trinhdo = pd.read_sql(query_trinhdo, conn_libol)
print(df_trinhdo)

   trinh_do_id                 Trinh_do
0           10                   PGS.TS
1            3                 Cao đẳng
2            4                  Đại học
3            5                  Thạc sĩ
4            6                  Tiến sĩ
5            7              Phó tiến sĩ
6           11  Trung học chuyên nghiệp
7            9             Trung học PT
8           12                    12/12


C:\Users\admin\AppData\Local\Temp\ipykernel_10892\4146533011.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trinhdo = pd.read_sql(query_trinhdo, conn_libol)


## Xử lý data

In [5]:
new_row = pd.DataFrame({'trinh_do_id': [0], 'Trinh_do': ['(Không xác định)']}) # Thêm 1 dòng không xác định
df_trinhdo = pd.concat([df_trinhdo, new_row], ignore_index=True)
df_trinhdo = df_trinhdo.sort_values(by='trinh_do_id', ascending=True).reset_index(drop=True)
print(df_trinhdo)

   trinh_do_id                 Trinh_do
0            0         (Không xác định)
1            3                 Cao đẳng
2            4                  Đại học
3            5                  Thạc sĩ
4            6                  Tiến sĩ
5            7              Phó tiến sĩ
6            9             Trung học PT
7           10                   PGS.TS
8           11  Trung học chuyên nghiệp
9           12                    12/12


## Load data

### [Nếu cần] Clear bảng

In [6]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Trinh_do"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim

In [7]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Trinh_do (ID_trinh_do, Loai_trinh_do) 
                VALUES (?, ?)
                """
for index, row in df_trinhdo.iterrows():
    values = (row['trinh_do_id'], 
              row['Trinh_do'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()